In [1]:
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.validation import explain_validity

# Shapely 2.x: make_valid (optional)
try:
    from shapely.validation import make_valid  # type: ignore
    HAS_MAKE_VALID = True
except Exception:
    HAS_MAKE_VALID = False


# =============================================================================
# Task 2 — Boundary QA (ADM2 polygons)
# Outputs (all prefixed with task2_):
#   outputs/ : JSON
#   tables/  : CSVs
#   outputs/ : issues_features.gpkg (if any issues)
# =============================================================================

# ----------------------------
# Project structure
# ----------------------------
ROOT = Path(r"C:\Users\am636\copperbelt_worldpop_smod")

DATA = ROOT / "data_raw"
OUT = ROOT / "outputs"
TABDIR = ROOT / "tables"
FIGDIR = ROOT / "figures"  # reserved (not used unless a figure is added later)

for d in (DATA, OUT, TABDIR, FIGDIR):
    d.mkdir(parents=True, exist_ok=True)

PREFIX = "task2_"

boundary_path = DATA / "ZMB_adm2_gadm41_Copperbelt.shp"

# Optional rule: expected ADM1 name (set to None to skip)
expected_name1 = "Copperbelt"  # or None

# Equal-area CRS for area diagnostics
equal_area_crs = "EPSG:6933"

missing = [p for p in (boundary_path,) if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input file in data_raw:\n" + "\n".join(str(p) for p in missing))

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


# ----------------------------
# Output paths
# ----------------------------
json_out = OUT / f"{PREFIX}qa_summary.json"
csv_out = TABDIR / f"{PREFIX}qa_summary.csv"
gpkg_out = OUT / f"{PREFIX}issues_features.gpkg"

dup_id_csv = TABDIR / f"{PREFIX}duplicates_by_id.csv"
dup_geom_csv = TABDIR / f"{PREFIX}duplicates_by_geometry.csv"
invalid_csv = TABDIR / f"{PREFIX}invalid_geometries.csv"
outscope_csv = TABDIR / f"{PREFIX}out_of_scope.csv"
overlaps_csv = TABDIR / f"{PREFIX}overlaps.csv"
gaps_csv = TABDIR / f"{PREFIX}gaps.csv"
slivers_csv = TABDIR / f"{PREFIX}slivers.csv"


# ----------------------------
# QA recording helpers
# ----------------------------
def status_rank(s: str) -> int:
    return {"PASS": 0, "WARN": 1, "FAIL": 2}.get(s, 1)

def worst_status(statuses) -> str:
    return max(statuses, key=status_rank) if statuses else "PASS"

checks = []

def add_check(name: str, status: str, metrics=None, note: str = ""):
    checks.append({
        "check": name,
        "status": status,
        "metrics": metrics or {},
        "note": note
    })

def safe_str(x) -> str:
    try:
        return str(x)
    except Exception:
        return repr(x)

def has_weird_encoding(series: pd.Series) -> bool:
    if series.dtype != "object":
        return False
    s = series.dropna().astype(str)
    if s.empty:
        return False
    return bool(s.str.contains("�", regex=False).any())

def geom_vertex_count(geom) -> int:
    if geom is None or geom.is_empty:
        return 0
    gt = geom.geom_type
    if gt == "Polygon":
        n = len(geom.exterior.coords) if geom.exterior else 0
        for ring in geom.interiors:
            n += len(ring.coords)
        return n
    if gt == "MultiPolygon":
        return sum(geom_vertex_count(g) for g in geom.geoms)
    return 0

def geometry_hash_md5_wkb(geom):
    if geom is None or geom.is_empty:
        return None
    return hashlib.md5(geom.wkb).hexdigest()

def try_fix_geometry(geom):
    if geom is None or geom.is_empty:
        return geom, None
    if geom.is_valid:
        return geom, None

    if HAS_MAKE_VALID:
        try:
            g2 = make_valid(geom)
            if g2 is not None and (not g2.is_empty) and g2.is_valid:
                return g2, "make_valid"
        except Exception:
            pass

    try:
        g2 = geom.buffer(0)
        if g2 is not None and (not g2.is_empty) and g2.is_valid:
            return g2, "buffer(0)"
    except Exception:
        pass

    return geom, None

def union_hole_areas(union_geom):
    if union_geom is None or union_geom.is_empty:
        return 0, 0.0

    hole_count = 0
    hole_area = 0.0

    def poly_holes(poly: Polygon):
        nonlocal hole_count, hole_area
        for ring in poly.interiors:
            hole_count += 1
            hole_area += Polygon(ring).area

    gt = union_geom.geom_type
    if gt == "Polygon":
        poly_holes(union_geom)
    elif gt == "MultiPolygon":
        for p in union_geom.geoms:
            poly_holes(p)

    return hole_count, float(hole_area)


# ----------------------------
# A) Read checks
# ----------------------------
add_check("A1 File exists", "PASS", metrics={"path": str(boundary_path)})

try:
    gdf = gpd.read_file(boundary_path)
    add_check("A2 Readable layer", "PASS", metrics={"features": int(len(gdf))})
except Exception as e:
    add_check("A2 Readable layer", "FAIL", metrics={"error": safe_str(e)})
    pd.DataFrame(checks).to_csv(csv_out, index=False)
    with open(json_out, "w", encoding="utf-8") as f:
        json.dump({"run_time_utc": run_time_utc, "checks": checks}, f, indent=2)
    raise

geom_types = sorted(set(gdf.geom_type.dropna().unique().tolist()))
unexpected = [t for t in geom_types if t not in ["Polygon", "MultiPolygon"]]
add_check(
    "A3 Geometry types are polygon-only",
    "FAIL" if unexpected else "PASS",
    metrics={"geom_types": geom_types, "unexpected": unexpected},
    note="Non-polygon geometries are not suitable for polygon zonal statistics." if unexpected else ""
)

obj_cols = [c for c in gdf.columns if gdf[c].dtype == "object" and c != "geometry"]
bad_cols = [c for c in obj_cols if has_weird_encoding(gdf[c])]
add_check(
    "A4 Text encoding heuristic",
    "WARN" if bad_cols else "PASS",
    metrics={"flagged_columns": bad_cols, "checked_object_columns": len(obj_cols)},
    note="Replacement characters found in some text fields." if bad_cols else ""
)


# ----------------------------
# B) CRS checks
# ----------------------------
crs = gdf.crs
if crs is None:
    add_check("B1 CRS exists", "FAIL", metrics={"crs": None}, note="CRS is missing.")
    crs_is_geographic = None
    crs_epsg = None
else:
    try:
        crs_epsg = crs.to_epsg()
    except Exception:
        crs_epsg = None
    add_check("B1 CRS exists", "PASS", metrics={"crs": safe_str(crs), "epsg": crs_epsg})
    crs_is_geographic = bool(crs.is_geographic)

if crs is not None and crs_is_geographic:
    add_check(
        "B2 Geographic CRS note",
        "WARN",
        metrics={"is_geographic": True},
        note=f"Area/length are not meaningful unless projected (e.g., {equal_area_crs})."
    )
elif crs is not None:
    add_check("B2 Geographic CRS note", "PASS", metrics={"is_geographic": False})


# ----------------------------
# C) Schema checks
# ----------------------------
col_dtypes = {c: safe_str(gdf[c].dtype) for c in gdf.columns if c != "geometry"}
add_check("C1 Schema overview", "PASS", metrics={"columns": list(gdf.columns), "dtypes": col_dtypes})

likely_keys = ["GID_0", "GID_1", "GID_2", "NAME_1", "NAME_2"]
missing_keys = [k for k in likely_keys if k not in gdf.columns]
add_check(
    "C2 Key fields present (soft check)",
    "WARN" if missing_keys else "PASS",
    metrics={"missing": missing_keys, "present": [k for k in likely_keys if k in gdf.columns]}
)

null_stats = {}
for c in [k for k in likely_keys if k in gdf.columns]:
    s = gdf[c]
    n_null = int(s.isna().sum())
    n_empty = int((s.astype(str).str.strip() == "").sum()) if s.dtype == "object" else 0
    null_stats[c] = {"null": n_null, "empty_string": n_empty}
any_problem = any(v["null"] > 0 or v["empty_string"] > 0 for v in null_stats.values()) if null_stats else True
add_check(
    "C3 Null/empty counts for key fields",
    "WARN" if any_problem else "PASS",
    metrics=null_stats,
    note="Key fields have missing/empty values." if any_problem and null_stats else ""
)

if "GID_2" in gdf.columns:
    gid2 = gdf["GID_2"].astype(str)
    dup_mask = gid2.duplicated(keep=False)
    n_dup = int(dup_mask.sum())
    if n_dup > 0:
        dup_vals = sorted(gid2[dup_mask].unique().tolist())
        add_check(
            "C4 Uniqueness of GID_2",
            "FAIL",
            metrics={"duplicates_count": n_dup, "duplicate_values": dup_vals[:50], "truncated": len(dup_vals) > 50},
            note="Duplicate IDs can break joins and distort zonal summaries."
        )
        dup_df = gdf.loc[dup_mask, ["GID_2"] + ([c for c in ["NAME_1", "NAME_2"] if c in gdf.columns])]
        dup_df = dup_df.assign(row_index=dup_df.index.values)
        dup_df.to_csv(dup_id_csv, index=False)
    else:
        add_check("C4 Uniqueness of GID_2", "PASS", metrics={"duplicates_count": 0})
else:
    add_check("C4 Uniqueness of stable ID", "WARN", note="No GID_2 field found.")

if "NAME_1" in gdf.columns:
    uniq_name1 = sorted(gdf["NAME_1"].dropna().astype(str).unique().tolist())
    add_check(
        "C5 Province consistency (unique NAME_1 values)",
        "PASS" if len(uniq_name1) <= 1 else "WARN",
        metrics={"unique_NAME_1": uniq_name1}
    )

    if expected_name1:
        mismatch = gdf["NAME_1"].astype(str) != expected_name1
        n_mismatch = int(mismatch.sum())
        if n_mismatch > 0:
            add_check(
                "C6 Out-of-scope by expected NAME_1",
                "FAIL",
                metrics={"expected_NAME_1": expected_name1, "mismatch_count": n_mismatch},
                note="Out-of-scope features should be removed or reported separately."
            )
            cols = [c for c in ["GID_2", "NAME_1", "NAME_2"] if c in gdf.columns]
            out_df = gdf.loc[mismatch, cols].copy()
            out_df["row_index"] = out_df.index.values
            out_df.to_csv(outscope_csv, index=False)
        else:
            add_check("C6 Out-of-scope by expected NAME_1", "PASS", metrics={"mismatch_count": 0})
else:
    add_check("C5 Province consistency (unique NAME_1 values)", "WARN", note="NAME_1 field not present.")
    if expected_name1:
        add_check("C6 Out-of-scope by expected NAME_1", "WARN", metrics={"expected_NAME_1": expected_name1})

if "NAME_1" in gdf.columns and "NAME_2" in gdf.columns:
    mapping = gdf.groupby(gdf["NAME_2"].astype(str))["NAME_1"].nunique(dropna=True)
    bad = mapping[mapping > 1]
    add_check(
        "C7 Hierarchy: NAME_2 maps to a single NAME_1",
        "WARN" if len(bad) > 0 else "PASS",
        metrics={"problematic_NAME_2_count": int(len(bad)), "examples": bad.head(10).to_dict() if len(bad) else {}}
    )
else:
    add_check("C7 Hierarchy: NAME_2 maps to a single NAME_1", "WARN", note="NAME_1/NAME_2 missing.")


# ----------------------------
# D) Geometry validity checks
# ----------------------------
geom = gdf.geometry
n_missing_geom = int(geom.isna().sum())
n_empty_geom = int(geom.is_empty.sum()) if hasattr(geom, "is_empty") else 0
valid_mask = geom.is_valid.fillna(False)
n_invalid = int((~valid_mask).sum()) - n_missing_geom
invalid_idx = gdf.index[(~valid_mask) & (~geom.isna())].tolist()

if n_missing_geom > 0 or n_empty_geom > 0 or n_invalid > 0:
    status = "FAIL" if (n_missing_geom > 0 or n_empty_geom > 0) else "WARN"
    add_check(
        "D1 Missing/empty/invalid geometries",
        status,
        metrics={"missing_geom": n_missing_geom, "empty_geom": n_empty_geom, "invalid_geom": int(n_invalid)}
    )
else:
    add_check("D1 Missing/empty/invalid geometries", "PASS", metrics={"missing_geom": 0, "empty_geom": 0, "invalid_geom": 0})

if n_invalid > 0:
    invalid_rows = []
    fixable = 0
    fix_methods = {"make_valid": 0, "buffer(0)": 0}

    for idx in invalid_idx:
        g = gdf.at[idx, "geometry"]
        reason = explain_validity(g)
        g2, method = try_fix_geometry(g)
        can_fix = bool(g2 is not None and (not g2.is_empty) and g2.is_valid)
        if can_fix:
            fixable += 1
            if method:
                fix_methods[method] = fix_methods.get(method, 0) + 1

        rec = {"row_index": idx, "reason": reason, "fixable": can_fix, "fix_method": method}
        for c in ["GID_2", "NAME_2", "NAME_1"]:
            if c in gdf.columns:
                rec[c] = gdf.at[idx, c]
        invalid_rows.append(rec)

    pd.DataFrame(invalid_rows).to_csv(invalid_csv, index=False)

    add_check(
        "D2 Invalid geometry diagnostics",
        "WARN" if fixable > 0 else "FAIL",
        metrics={"invalid_count": int(n_invalid), "fixable_estimate": int(fixable), "fix_methods": fix_methods, "csv": str(invalid_csv)}
    )
else:
    add_check("D2 Invalid geometry diagnostics", "PASS", metrics={"invalid_count": 0})


# ----------------------------
# E) Duplicate geometry checks
# ----------------------------
gdf["_geom_hash"] = [geometry_hash_md5_wkb(g) for g in gdf.geometry]

dup_geom_mask = gdf["_geom_hash"].notna() & gdf["_geom_hash"].duplicated(keep=False)
n_dup_geom = int(dup_geom_mask.sum())

if n_dup_geom > 0:
    groups = (gdf.loc[dup_geom_mask]
              .groupby("_geom_hash")
              .apply(lambda x: x.index.tolist())
              .to_dict())

    add_check(
        "E1 Duplicate geometries (exact)",
        "FAIL",
        metrics={"duplicate_groups": int(len(groups)), "features_involved": int(n_dup_geom)}
    )

    rows = []
    for h, idxs in groups.items():
        for idx in idxs:
            rec = {"geom_hash": h, "row_index": idx}
            for c in ["GID_2", "NAME_2", "NAME_1"]:
                if c in gdf.columns:
                    rec[c] = gdf.at[idx, c]
            rows.append(rec)

    pd.DataFrame(rows).to_csv(dup_geom_csv, index=False)
else:
    add_check("E1 Duplicate geometries (exact)", "PASS", metrics={"duplicate_groups": 0, "features_involved": 0})


# ----------------------------
# F) Topology diagnostics
# ----------------------------
if gdf.crs is not None:
    try:
        gdf_ea = gdf.to_crs(equal_area_crs)
        ea_ok = True
        add_check("F0 Equal-area projection", "PASS", metrics={"equal_area_crs": equal_area_crs})
    except Exception as e:
        gdf_ea = None
        ea_ok = False
        add_check("F0 Equal-area projection", "WARN", metrics={"error": safe_str(e), "equal_area_crs": equal_area_crs})
else:
    gdf_ea = None
    ea_ok = False
    add_check("F0 Equal-area projection", "WARN", metrics={"equal_area_crs": equal_area_crs}, note="Missing CRS.")


# Overlaps (spatial index)
overlap_pairs = []
try:
    sindex = gdf.sindex
    idx_list = list(gdf.index)

    for i in idx_list:
        gi = gdf.at[i, "geometry"]
        if gi is None or gi.is_empty:
            continue

        cand = list(sindex.intersection(gi.bounds))
        for j in cand:
            if j <= i:
                continue

            gj = gdf.at[j, "geometry"]
            if gj is None or gj.is_empty:
                continue

            if not gi.intersects(gj):
                continue
            if gi.touches(gj):
                continue

            rec = {"i": i, "j": j}
            for c in ["GID_2", "NAME_2"]:
                if c in gdf.columns:
                    rec[f"{c}_i"] = gdf.at[i, c]
                    rec[f"{c}_j"] = gdf.at[j, c]

            if ea_ok:
                gi_ea = gdf_ea.at[i, "geometry"]
                gj_ea = gdf_ea.at[j, "geometry"]
                rec["overlap_area_m2"] = float(gi_ea.intersection(gj_ea).area)

            overlap_pairs.append(rec)

    if overlap_pairs:
        add_check("F1 Overlaps between polygons", "FAIL", metrics={"overlap_pair_count": int(len(overlap_pairs)), "csv": str(overlaps_csv)})
        pd.DataFrame(overlap_pairs).to_csv(overlaps_csv, index=False)
    else:
        add_check("F1 Overlaps between polygons", "PASS", metrics={"overlap_pair_count": 0})

except Exception as e:
    add_check("F1 Overlaps between polygons", "WARN", metrics={"error": safe_str(e)})
    overlap_pairs = None


# Enclosed holes in union (equal-area)
if ea_ok:
    union = gdf_ea.geometry.unary_union
    hole_count, hole_area = union_hole_areas(union)
    union_area = float(union.area) if union and (not union.is_empty) else 0.0
    hole_ratio = (hole_area / union_area) if union_area > 0 else 0.0

    if hole_count == 0:
        add_check("F2 Enclosed gaps (holes inside union)", "PASS", metrics={"hole_count": 0, "hole_area_m2": 0.0, "hole_ratio": 0.0})
    else:
        status = "WARN" if hole_ratio < 0.001 else "FAIL"
        add_check("F2 Enclosed gaps (holes inside union)", status, metrics={"hole_count": int(hole_count), "hole_area_m2": float(hole_area), "hole_ratio": float(hole_ratio), "csv": str(gaps_csv)})
        pd.DataFrame([{
            "hole_count": hole_count,
            "hole_area_m2": hole_area,
            "hole_ratio": hole_ratio
        }]).to_csv(gaps_csv, index=False)
else:
    add_check("F2 Enclosed gaps (holes inside union)", "WARN", note="Requires equal-area projection.")


# Slivers (equal-area)
if ea_ok:
    areas = gdf_ea.geometry.area
    median_area = float(np.nanmedian(areas.values)) if len(areas) else 0.0
    thresh = min(0.005 * median_area, 1e6) if median_area > 0 else 0.0

    if thresh > 0:
        sliver_mask = areas < thresh
        n_slivers = int(sliver_mask.sum())
    else:
        sliver_mask = np.array([False] * len(gdf_ea))
        n_slivers = 0

    if n_slivers > 0:
        add_check("F3 Slivers (very small polygons)", "WARN", metrics={"sliver_count": n_slivers, "threshold_m2": float(thresh), "median_area_m2": median_area, "csv": str(slivers_csv)})
        cols = [c for c in ["GID_2", "NAME_2", "NAME_1"] if c in gdf.columns]
        sl_df = gdf.loc[sliver_mask, cols].copy()
        sl_df["row_index"] = sl_df.index.values
        sl_df["area_m2"] = areas[sliver_mask].astype(float).values
        sl_df.to_csv(slivers_csv, index=False)
    else:
        add_check("F3 Slivers (very small polygons)", "PASS", metrics={"sliver_count": 0, "threshold_m2": float(thresh), "median_area_m2": median_area})
else:
    add_check("F3 Slivers (very small polygons)", "WARN", note="Requires equal-area projection.")


# ----------------------------
# G) Zonal readiness checks
# ----------------------------
vertex_counts = gdf.geometry.apply(geom_vertex_count)
multipart_mask = gdf.geom_type == "MultiPolygon"
n_multipart = int(multipart_mask.sum())

complex_metrics = {
    "features": int(len(gdf)),
    "multipart_count": n_multipart,
    "vertex_count_min": int(vertex_counts.min()) if len(vertex_counts) else None,
    "vertex_count_median": float(np.median(vertex_counts.values)) if len(vertex_counts) else None,
    "vertex_count_p95": float(np.percentile(vertex_counts.values, 95)) if len(vertex_counts) else None,
    "vertex_count_max": int(vertex_counts.max()) if len(vertex_counts) else None,
}

if len(vertex_counts) and complex_metrics["vertex_count_p95"] and complex_metrics["vertex_count_p95"] > 20000:
    add_check("G1 Geometry complexity", "WARN", metrics=complex_metrics)
else:
    add_check("G1 Geometry complexity", "PASS", metrics=complex_metrics)

add_check(
    "G2 Zonal statistics readiness notes",
    "PASS",
    metrics={
        "notes": [
            "Overlaps can double-count population in zonal sums.",
            "Invalid geometries can cause failures or incorrect results in zonal operations.",
            "Out-of-scope features should be excluded or reported separately."
        ]
    }
)


# ----------------------------
# H) Issues-only layer (GeoPackage)
# ----------------------------
issue_map = {idx: [] for idx in gdf.index}

if "GID_2" in gdf.columns:
    gid2 = gdf["GID_2"].astype(str)
    dup_id_mask = gid2.duplicated(keep=False)
    for idx in gdf.index[dup_id_mask]:
        issue_map[idx].append("DUPLICATE_GID_2")

if n_dup_geom > 0:
    for idx in gdf.index[dup_geom_mask]:
        issue_map[idx].append("DUPLICATE_GEOMETRY")

if n_invalid > 0:
    for idx in invalid_idx:
        issue_map[idx].append("INVALID_GEOMETRY")

if expected_name1 and "NAME_1" in gdf.columns:
    mismatch = gdf["NAME_1"].astype(str) != expected_name1
    for idx in gdf.index[mismatch]:
        issue_map[idx].append("OUT_OF_SCOPE_NAME_1")

if overlap_pairs is not None and overlap_pairs:
    for rec in overlap_pairs:
        issue_map[rec["i"]].append("OVERLAPS_OTHER_POLYGON")
        issue_map[rec["j"]].append("OVERLAPS_OTHER_POLYGON")

if ea_ok and "sliver_mask" in locals() and sliver_mask is not None:
    for idx in gdf.index[sliver_mask]:
        issue_map[idx].append("SLIVER_SMALL_AREA")

flagged = [idx for idx, issues in issue_map.items() if issues]

if flagged:
    issues_gdf = gdf.loc[flagged].copy()
    issues_gdf["issues"] = [(";".join(sorted(set(issue_map[idx])))) for idx in issues_gdf.index]
    issues_gdf["issue_count"] = issues_gdf["issues"].apply(lambda x: 0 if not x else len(x.split(";")))
    issues_gdf.to_file(gpkg_out, layer="issues_features", driver="GPKG")
    add_check("H1 Issues layer written", "PASS", metrics={"flagged_feature_count": int(len(issues_gdf)), "gpkg": str(gpkg_out)})
else:
    add_check("H1 Issues layer written", "PASS", metrics={"flagged_feature_count": 0})


# ----------------------------
# Write summary outputs
# ----------------------------
summary = {
    "run_time_utc": run_time_utc,
    "boundary_path": str(boundary_path),
    "project_root": str(ROOT),
    "expected_name1": expected_name1,
    "equal_area_crs": equal_area_crs,
    "checks": checks,
}

csv_rows = [{
    "check": c["check"],
    "status": c["status"],
    "note": c["note"],
    "metrics_json": json.dumps(c["metrics"], ensure_ascii=False),
} for c in checks]

pd.DataFrame(csv_rows).to_csv(csv_out, index=False)
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)


# ----------------------------
# Console summary
# ----------------------------
status_counts = pd.Series([c["status"] for c in checks]).value_counts().to_dict()
overall = worst_status([c["status"] for c in checks if not c["check"].startswith("H1")])

def find_check(name):
    for c in checks:
        if c["check"] == name:
            return c
    return None

c_gid = find_check("C4 Uniqueness of GID_2")
c_geomdup = find_check("E1 Duplicate geometries (exact)")
c_invalid = find_check("D1 Missing/empty/invalid geometries")
c_outscope = find_check("C6 Out-of-scope by expected NAME_1")
c_overlap = find_check("F1 Overlaps between polygons")
c_holes = find_check("F2 Enclosed gaps (holes inside union)")

print("\n================= BOUNDARY QA SUMMARY (TASK 2) =================")
print("Run time (UTC):", run_time_utc)
print("Input:", boundary_path.name)
print("Overall status:", overall)
print("Status counts:", status_counts)
print("Saved:")
print(" -", csv_out)
print(" -", json_out)
if gpkg_out.exists():
    print(" -", gpkg_out)

print("\nKey findings:")
if c_gid:
    print(f"• GID_2 uniqueness: {c_gid['status']}")
    if c_gid["status"] != "PASS" and dup_id_csv.exists():
        print("  -", dup_id_csv)

if c_geomdup:
    print(f"• Duplicate geometries: {c_geomdup['status']}")
    if c_geomdup["status"] != "PASS" and dup_geom_csv.exists():
        print("  -", dup_geom_csv)

if c_invalid:
    print(f"• Geometry validity: {c_invalid['status']} | {c_invalid['metrics']}")
    if n_invalid > 0 and invalid_csv.exists():
        print("  -", invalid_csv)

if c_outscope:
    print(f"• Out-of-scope by NAME_1: {c_outscope['status']}")
    if c_outscope["status"] != "PASS" and outscope_csv.exists():
        print("  -", outscope_csv)

if c_overlap:
    print(f"• Overlaps: {c_overlap['status']}")
    if c_overlap["status"] != "PASS" and overlaps_csv.exists():
        print("  -", overlaps_csv)

if c_holes:
    print(f"• Enclosed gaps (holes): {c_holes['status']}")
    if c_holes["status"] != "PASS" and gaps_csv.exists():
        print("  -", gaps_csv)

print("===============================================================\n")

if "_geom_hash" in gdf.columns:
    gdf.drop(columns=["_geom_hash"], inplace=True)



================= BOUNDARY QA SUMMARY (TASK 2) =================
Run time (UTC): 2026-01-30T04:17:39+00:00
Input: ZMB_adm2_gadm41_Copperbelt.shp
Overall status: FAIL
Status counts: {'PASS': 17, 'FAIL': 4, 'WARN': 2}
Saved:
 - C:\Users\am636\copperbelt_worldpop_smod\tables\task2_qa_summary.csv
 - C:\Users\am636\copperbelt_worldpop_smod\outputs\task2_qa_summary.json
 - C:\Users\am636\copperbelt_worldpop_smod\outputs\task2_issues_features.gpkg

Key findings:
• GID_2 uniqueness: FAIL
  - C:\Users\am636\copperbelt_worldpop_smod\tables\task2_duplicates_by_id.csv
• Duplicate geometries: FAIL
  - C:\Users\am636\copperbelt_worldpop_smod\tables\task2_duplicates_by_geometry.csv
• Geometry validity: PASS | {'missing_geom': 0, 'empty_geom': 0, 'invalid_geom': 0}
• Out-of-scope by NAME_1: FAIL
  - C:\Users\am636\copperbelt_worldpop_smod\tables\task2_out_of_scope.csv
• Overlaps: FAIL
  - C:\Users\am636\copperbelt_worldpop_smod\tables\task2_overlaps.csv
• Enclosed gaps (holes): PASS



C:\Users\am636\AppData\Local\Temp\ipykernel_1620\2168676980.py:476: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  union = gdf_ea.geometry.unary_union
